In [1]:
import torch, torch.nn.functional as F, glob, json

outputs = "/n/holylfs06/LABS/kempner_project_b/Lab/dendritic/adivm/dendritic-modeling/outputs"

# load ne2048 noproj as representative run
run_dir = sorted(glob.glob(f"{outputs}/alexnet_k1_ne2048_noproj_ddp_*"))[-1]
state = torch.load(f"{run_dir}/final_model.pt", map_location="cpu")
sd = state.get("model_state_dict", state)

# --- 1. check synaptic weights (from saved checkpoint, no data needed) ---
print("=== Synaptic weights (softplus applied) ===")
for k, v in sd.items():
    if "log_weight" in k:
        w = F.softplus(v)
        print(f"{k}")
        print(f"  min={w.min():.6f}  max={w.max():.4f}  all_positive={bool((w > 0).all())}")

# --- 2. check encoder output activations (needs a forward pass) ----------
# The encoder ends with ReLU(fc6), so outputs are non-negative by construction.
# Verify on a small batch of real images:
import torchvision.models as tvm
from dendritic_modeling.datasets.standard_datasets import load_imagenet_as_datasets
from torch.utils.data import DataLoader

_, _, test_ds = load_imagenet_as_datasets(
    train_valid_split=0.8, flatten=False, normalize=True,
    task_data_path="/n/holylfs06/LABS/kempner_shared/Everyone/testbed/vision/imagenet_1k"
)
loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4)
inputs, _ = next(iter(loader))

model = tvm.alexnet(weights=tvm.AlexNet_Weights.IMAGENET1K_V1)
# encoder = features + avgpool + fc6 + ReLU (everything up to and including classifier.2)
encoder = torch.nn.Sequential(
    model.features,
    model.avgpool,
    torch.nn.Flatten(1),
    model.classifier[1],  # fc6 Linear
    model.classifier[2],  # fc6 ReLU
)
encoder.eval()
with torch.no_grad():
    enc_out = encoder(inputs)

print("\n=== Encoder output activations (input to EINet) ===")
print(f"  min={enc_out.min():.6f}  max={enc_out.max():.4f}")
print(f"  all_non_negative={bool((enc_out >= 0).all())}")
print(f"  fraction zero (dead ReLU): {(enc_out == 0).float().mean():.3f}")


/tmp/ipykernel_239873/4049246883.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(f"{run_dir}/final_model.pt", map_location="cpu")


=== Synaptic weights (softplus applied) ===
core_network.core.layers.0.inhibitory_cells.branch_layers.1.branches_to_output.log_weight
  min=0.000001  max=0.0000  all_positive=True
core_network.core.layers.0.inhibitory_cells.branch_layers.2.branches_to_output.log_weight
  min=24.125904  max=24.9226  all_positive=True
core_network.core.layers.0.excitatory_cells.branch_layers.1.branches_to_output.log_weight
  min=14.562120  max=15.3973  all_positive=True
core_network.core.layers.0.excitatory_cells.branch_layers.2.branches_to_output.log_weight
  min=24.099186  max=25.3176  all_positive=True


: 